# Pizza-Bot - Pour aller plus loin

Trois prolongements du projet :

1. ajouter l'Hôpital comme troisième emplacement, sans modifier les fonctions de la partie 1 ;
2. régler la persistance, de 1 à 5, et mesurer ce que chaque réglage coûte ;
3. remplacer la référence fixe par une référence glissante, et voir ce qui se passe quand la
   buse dérive très lentement.

Le programme complet, lui, est dans le fichier `pizza_bot.py`.

Règles du projet : Python de base, aucun import.

## 1. Trois emplacements

Les fonctions de la partie 1 parcourent la liste des emplacements au lieu d'écrire « Campus » et
« Gare » en dur. On peut donc leur passer une liste de trois noms sans changer une ligne.

Gains inventés dans le même esprit que la matrice d'origine : deux distributeurs au même endroit
se partagent la clientèle, et l'Hôpital est un emplacement moyen, moins rentable que le Campus
mais plus régulier que la Gare.

In [1]:
EMPLACEMENTS = ["Campus", "Gare", "Hopital"]

GAINS = {
    ("Campus", "Campus"): (4, 4), ("Campus", "Gare"): (9, 7), ("Campus", "Hopital"): (8, 5),
    ("Gare", "Campus"): (6, 8), ("Gare", "Gare"): (3, 3), ("Gare", "Hopital"): (7, 5),
    ("Hopital", "Campus"): (5, 8), ("Hopital", "Gare"): (5, 7), ("Hopital", "Hopital"): (2, 2),
}

def est_equilibre_nash(gains, choix_pizzabot, choix_mamma, emplacements):
    """True si aucun joueur ne gagne strictement plus en changeant seul de stratégie."""
    gain_pizzabot = gains[(choix_pizzabot, choix_mamma)][0]
    gain_mamma = gains[(choix_pizzabot, choix_mamma)][1]

    for autre in emplacements:
        if gains[(autre, choix_mamma)][0] > gain_pizzabot:
            return False
    for autre in emplacements:
        if gains[(choix_pizzabot, autre)][1] > gain_mamma:
            return False
    return True

def trouver_equilibres_nash(gains, emplacements):
    equilibres = []
    for choix_pizzabot in emplacements:
        for choix_mamma in emplacements:
            if est_equilibre_nash(gains, choix_pizzabot, choix_mamma, emplacements):
                equilibres.append((choix_pizzabot, choix_mamma))
    return equilibres

# La matrice complète, pour lire les gains case par case.
print("Matrice des gains (Pizza-Bot ; Mamma-Auto)")
print(f"{'':10}", end="")
for choix_mamma in EMPLACEMENTS:
    print(f"{choix_mamma:>12}", end="")
print()
for choix_pizzabot in EMPLACEMENTS:
    print(f"{choix_pizzabot:10}", end="")
    for choix_mamma in EMPLACEMENTS:
        case = GAINS[(choix_pizzabot, choix_mamma)]
        print(f"{str(case[0]) + ' ; ' + str(case[1]):>12}", end="")
    print()

print()
for case in trouver_equilibres_nash(GAINS, EMPLACEMENTS):
    print(f"Équilibre de Nash : {case} -> gains {GAINS[case]}")

Matrice des gains (Pizza-Bot ; Mamma-Auto)
                Campus        Gare     Hopital
Campus           4 ; 4       9 ; 7       8 ; 5
Gare             6 ; 8       3 ; 3       7 ; 5
Hopital          5 ; 8       5 ; 7       2 ; 2

Équilibre de Nash : ('Campus', 'Gare') -> gains (9, 7)
Équilibre de Nash : ('Gare', 'Campus') -> gains (6, 8)


**Lecture du résultat.** Les fonctions fonctionnent sans aucune modification : elles ne
connaissent pas le nombre d'emplacements, elles parcourent la liste qu'on leur donne. Avec ces
gains, les équilibres restent (Campus ; Gare) et (Gare ; Campus). L'Hôpital n'apparaît dans aucun
équilibre : il rapporte toujours moins que la meilleure alternative, donc chaque joueur préfère
en partir. On dit qu'il est dominé.

Si l'on voulait le voir apparaître, il faudrait lui donner des gains plus élevés, par exemple en
imaginant une clientèle captive de personnel soignant.

## 2. Régler la persistance

On fait varier la persistance de 1 à 5 et on regarde, sur le flux de production, où la machine
s'arrête et combien de pizzas mal garnies ont été servies depuis le début de la dérive.

La dérive commence à la pesée 16, avec 103,2 g : c'est la première valeur de la montée continue,
même si elle n'a pas encore franchi le seuil.

In [2]:
historique_calibration = [
    102.5, 102.1, 96.9, 101.0, 98.9, 97.5, 100.3, 101.3, 104.1, 99.5,
    102.4, 93.2, 99.3, 99.9, 98.8, 101.9, 99.5, 101.9, 102.0, 98.0,
    99.6, 99.8, 99.6, 98.3, 98.4, 100.9, 98.2, 98.9, 101.6, 101.4,
    100.0, 97.6, 96.0, 100.2, 99.9, 101.9, 101.8, 99.1, 99.6, 100.0,
    100.2, 100.9, 102.7, 100.9, 100.6, 99.5, 98.5, 98.6, 98.7, 100.1,
]

flux_temps_reel = [
    100.4, 99.1, 101.7, 98.6, 100.9, 106.8, 99.8, 100.3, 97.9, 101.2,
    92.4, 100.6, 99.4, 102.3, 100.1, 98.8, 103.2, 104.1, 104.9, 105.6,
    106.3,
]

DEBUT_DERIVE = 16

def moyenne(valeurs):
    return sum(valeurs) / len(valeurs)

def ecart_type_estime(valeurs):
    m = moyenne(valeurs)
    total = 0
    for v in valeurs:
        total = total + (v - m) ** 2
    return (total / (len(valeurs) - 1)) ** 0.5

def z_score(valeur, mu, sigma):
    return (valeur - mu) / sigma

def est_anomalie(valeur, mu, sigma, k=2):
    return abs(z_score(valeur, mu, sigma)) > k

def surveiller(flux, mu, sigma, k=2, persistance=3):
    alertes = []
    consecutives = 0
    for indice, valeur in enumerate(flux):
        if est_anomalie(valeur, mu, sigma, k):
            alertes.append((indice, valeur))
            consecutives = consecutives + 1
            if consecutives == persistance:
                return (alertes, indice)
        else:
            consecutives = 0
    return (alertes, None)

mu = moyenne(historique_calibration)
sigma = ecart_type_estime(historique_calibration)

print("persistance | arret a la pesee | pizzas servies depuis le debut de la derive")
for persistance in range(1, 6):
    alertes, arret = surveiller(flux_temps_reel, mu, sigma, 2, persistance)
    if arret is None:
        print(f"     {persistance}      |      jamais      |  toutes : la derive n'est pas arretee")
    elif arret < DEBUT_DERIVE:
        print(f"     {persistance}      |        {arret:2}        |  arret sur un pic isole, avant la derive")
    else:
        print(f"     {persistance}      |        {arret:2}        |  {arret - DEBUT_DERIVE + 1}")

persistance | arret a la pesee | pizzas servies depuis le debut de la derive
     1      |         5        |  arret sur un pic isole, avant la derive
     2      |        18        |  3
     3      |        19        |  4
     4      |        20        |  5
     5      |      jamais      |  toutes : la derive n'est pas arretee


**Lecture du résultat.** Le tableau montre l'arbitrage en entier.

À persistance 1, la machine s'arrête à la pesée 5, sur un pic isolé, alors que la buse va bien :
la production est interrompue pour rien. À persistance 2, l'arrêt tombe à la pesée 18, soit trois
pizzas mal garnies. À persistance 3, notre réglage, quatre pizzas. À persistance 4, cinq. À
persistance 5, la machine ne s'arrête jamais : le flux se termine avant que cinq anomalies ne se
suivent, et la dérive passe entièrement inaperçue.

C'est l'arbitrage du seuil transposé au temps : plus on attend pour être sûr, plus on sert de
pizzas mal garnies, et au-delà d'un certain point on ne détecte plus rien. La persistance de 3
est un bon compromis ici : elle élimine les deux pics isolés et ne coûte que quatre pizzas.

## 3. Un seuil qui apprend

On remplace la référence fixe, calculée une fois pour toutes sur l'historique de calibration, par
une référence glissante calculée sur les cinquante dernières pesées.

Pour voir la différence, on fabrique un flux qui dérive très lentement : la buse ajoute 0,05 g à
chaque pesée, soit 10 g au bout de 200 pesées.

In [3]:
historique_calibration = [
    102.5, 102.1, 96.9, 101.0, 98.9, 97.5, 100.3, 101.3, 104.1, 99.5,
    102.4, 93.2, 99.3, 99.9, 98.8, 101.9, 99.5, 101.9, 102.0, 98.0,
    99.6, 99.8, 99.6, 98.3, 98.4, 100.9, 98.2, 98.9, 101.6, 101.4,
    100.0, 97.6, 96.0, 100.2, 99.9, 101.9, 101.8, 99.1, 99.6, 100.0,
    100.2, 100.9, 102.7, 100.9, 100.6, 99.5, 98.5, 98.6, 98.7, 100.1,
]

FENETRE = 50

def moyenne(valeurs):
    return sum(valeurs) / len(valeurs)

def ecart_type_estime(valeurs):
    m = moyenne(valeurs)
    total = 0
    for v in valeurs:
        total = total + (v - m) ** 2
    return (total / (len(valeurs) - 1)) ** 0.5

def est_anomalie(valeur, mu, sigma, k=2):
    return abs((valeur - mu) / sigma) > k

# Un flux qui monte de 0,05 g a chaque pesee.
flux_lent = []
valeur = 100.0
for i in range(200):
    flux_lent.append(round(valeur, 2))
    valeur = valeur + 0.05

# Reference fixe : mu et sigma de l'historique, qui ne bougent jamais.
mu_fixe = moyenne(historique_calibration)
sigma_fixe = ecart_type_estime(historique_calibration)

# Reference glissante : les cinquante dernieres pesees, mise a jour a chaque tour.
fenetre = list(historique_calibration)

alertes_fixes = 0
alertes_glissantes = 0

for valeur in flux_lent:
    if est_anomalie(valeur, mu_fixe, sigma_fixe):
        alertes_fixes = alertes_fixes + 1

    mu_glissant = moyenne(fenetre)
    sigma_glissant = ecart_type_estime(fenetre)
    if est_anomalie(valeur, mu_glissant, sigma_glissant):
        alertes_glissantes = alertes_glissantes + 1

    fenetre.append(valeur)
    if len(fenetre) > FENETRE:
        fenetre = fenetre[1:]

print(f"Flux : {len(flux_lent)} pesees, de {flux_lent[0]} g a {flux_lent[-1]} g")
print(f"Reference fixe      : {alertes_fixes} alertes")
print(f"Reference glissante : {alertes_glissantes} alertes")
print(f"Moyenne glissante finale : {moyenne(fenetre):.2f} g")

Flux : 200 pesees, de 100.0 g a 109.95 g
Reference fixe      : 126 alertes
Reference glissante : 0 alertes
Moyenne glissante finale : 108.72 g


**Lecture du résultat.** La référence fixe déclenche 126 alertes : elle voit la buse s'éloigner
de sa valeur de calibration. La référence glissante n'en déclenche aucune.

La raison est simple : elle se recalcule sur les cinquante dernières pesées, qui contiennent déjà
la dérive. Sa moyenne finit à 108,7 g, alors que la buse est censée déposer 100 g. Autrement dit,
le seuil a suivi la dérive et l'a acceptée comme le nouveau comportement normal.

C'est exactement le danger annoncé dans le sujet. Une référence glissante est utile quand la
machine change légitimement de régime, par exemple après un changement de fromage ou de réglage.
Mais elle ne doit jamais remplacer la référence fixe : on garde les deux, la référence fixe pour
détecter la dérive lente, la glissante pour éviter les fausses alertes dues aux variations
normales.

## Ce que ces trois prolongements montrent

Une fonction qui ne dépend que de ses arguments se réutilise sans être réécrite : c'est ce que
démontre l'ajout de l'Hôpital.

Un réglage de supervision ne se juge pas en sigmas ni en nombre d'anomalies, mais en conséquences
concrètes : combien d'arrêts inutiles, combien de pizzas mal garnies.

Enfin, une méthode qui s'adapte trop bien aux données finit par ne plus rien détecter. Il faut
toujours un point de comparaison fixe.